# Document Intelligence Platform - Colab eval runner

Runtime: **GPU (T4)**. Clones the repo, installs deps, runs Ollama (Qwen2.5-3B on GPU) +
an embedded Qdrant, executes the full pipeline over the 120-doc eval set, and produces
`artifacts_bundle.zip`.

**Runtime -> Run all.** ~30-40 min. If a cell fails, fix and Run all again (parsing is cached).

In [ ]:
REPO_URL = "https://github.com/PrayuktaKute/document_enterprise_platform.git"
import os
if not os.path.isdir('/content/document_enterprise_platform'):
    !git clone $REPO_URL /content/document_enterprise_platform
%cd /content/document_enterprise_platform

In [ ]:
!bash scripts/colab_bootstrap.sh

In [ ]:
# Start Ollama as a kernel-lived process (survives across cells) and pull the model.
import subprocess, time, urllib.request, os

os.environ["OLLAMA_NUM_PARALLEL"] = "4"
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"

def _ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        return True
    except Exception:
        return False

if not _ollama_up():
    subprocess.Popen(["ollama", "serve"],
                     stdout=open("/content/ollama.log", "w"),
                     stderr=subprocess.STDOUT,
                     env={**os.environ})
    for _ in range(60):
        if _ollama_up():
            break
        time.sleep(1)

assert _ollama_up(), "Ollama did not come up -- see /content/ollama.log"
print("ollama serving")
subprocess.run(["ollama", "pull", "qwen2.5:3b-instruct-q4_K_M"], check=True)
subprocess.run(["ollama", "list"], check=True)

### Data
If `data/raw` is not in the repo, regenerate it (SROIE + CUAD download, synthetic PDFs).

In [ ]:
import os
# manifest.jsonl is committed but the raw documents are not -- regenerate them.
if not os.path.isdir('data/raw/invoices') or not os.listdir('data/raw/invoices'):
    !python scripts/fetch_data.py --invoices 30 --contracts 30
    !python scripts/gen_synthetic.py --purchase-orders 30 --medical 30 --seed 7
    !python scripts/prepare_eval.py

In [ ]:
!WORKERS=4 bash scripts/run_all_eval.sh

In [ ]:
import json
print(open('artifacts/eval_report.md').read())
print(json.dumps(json.load(open('artifacts/retrieval_metrics.json')), indent=2)[:1500])

In [ ]:
from google.colab import files
files.download('artifacts_bundle.zip')